In [ ]:
import pandas as pd
import sqlite3

# الاتصال بقاعدة البيانات
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

print("🚀 Building Ultimate Matrix A with Advanced Feature Engineering...")

query_matrix_a = """
WITH 
-- 1. حساب مخرج المريض (Target)
PatientOutcome AS (
    SELECT 
        primaryid,
        MAX(CASE WHEN outc_cod IN ('DE', 'LT', 'HO', 'DS', 'CA', 'RI') THEN 1 ELSE 0 END) as is_severe
    FROM outc_clean
    GROUP BY primaryid
),

-- 2. جلب الدواء الأساسي (Primary Suspect) 
PrimaryDrug AS (
    SELECT 
        primaryid,
        MAX(final_drug_name) as primary_suspect_drug,
        MAX(route) as ps_route
    FROM drug_clean
    WHERE role_cod = 'PS' AND final_drug_name != 'none'
    GROUP BY primaryid
),

-- 3. Feature: Polypharmacy (عدد الأدوية لكل مريض)
Polypharmacy AS (
    SELECT 
        primaryid,
        COUNT(final_drug_name) as num_drugs
    FROM drug_clean
    GROUP BY primaryid
),

-- 4. Feature: Therapy Duration (أقصى فترة علاج بالدواء)
TherapyDuration AS (
    SELECT 
        primaryid,
        MAX(dur) as therapy_duration
    FROM ther_clean
    WHERE dur IS NOT NULL
    GROUP BY primaryid
),

-- 5. Feature: Report Source (مصدر التقرير من جدول RPSR)
ReportSource AS (
    SELECT 
        primaryid,
        MAX(rpsr_cod) as rpsr_cod
    FROM rpsr_clean
    GROUP BY primaryid
)

-- 6. الدمج النهائي للمصفوفة (Master Join)
SELECT 
    d.primaryid,
    d.age,
    d.wt,
    d.sex,
    d.occp_cod,
    d.rept_cod,
    d.is_test_set,
    p.primary_suspect_drug,
    p.ps_route,
    COALESCE(poly.num_drugs, 1) as num_drugs,
    t.therapy_duration,
    rs.rpsr_cod,
    COALESCE(o.is_severe, 0) as is_severe
FROM demo_clean d
INNER JOIN PrimaryDrug p ON d.primaryid = p.primaryid
LEFT JOIN Polypharmacy poly ON d.primaryid = poly.primaryid
LEFT JOIN TherapyDuration t ON d.primaryid = t.primaryid
LEFT JOIN ReportSource rs ON d.primaryid = rs.primaryid
LEFT JOIN PatientOutcome o ON d.primaryid = o.primaryid
WHERE d.age IS NOT NULL 
  AND p.primary_suspect_drug IS NOT NULL
"""

# تنفيذ الاستعلام
df_matrix_a = pd.read_sql_query(query_matrix_a, conn)

# حفظ الماتريكس في الداتابيز
matrix_name = 'matrix_a_outcome_v5'
df_matrix_a.to_sql(matrix_name, conn, if_exists='replace', index=False)

print(f"✅ Matrix A built successfully with ADVANCED FEATURES! Final Shape: {df_matrix_a.shape}")
display(df_matrix_a.head())

In [ ]:
import pandas as pd
import sqlite3
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# الاتصال بقاعدة البيانات
db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

print("🚀 Loading Matrix A (v5) and starting Ultimate ML Pipeline...")

# 1. قراءة الماتريكس الجديدة
df_ml = pd.read_sql_query("SELECT * FROM matrix_a_outcome_v5", conn)

# 2. التقسيم الزمني الصارم (Time-Based Split)
train_mask = df_ml['is_test_set'] == 0
test_mask = df_ml['is_test_set'] == 1

# فصل التدريب والاختبار
X_train = df_ml[train_mask].drop(columns=['is_severe', 'is_test_set', 'primaryid'])
y_train = df_ml[train_mask]['is_severe']

X_test = df_ml[test_mask].drop(columns=['is_severe', 'is_test_set', 'primaryid'])
y_test = df_ml[test_mask]['is_severe']

print(f"📦 Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

# 3. Target Encoding (بما في ذلك الميزة الجديدة rpsr_cod)
target_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod', 'rpsr_cod']
overall_mean = y_train.mean()

print("⚙️ Applying Target Encoding (Strictly on Train Data)...")
for col in target_cols:
    # حساب المتوسط من بيانات التدريب فقط (بنعوض عن الـ NaNs بـ 'UNK' مؤقتاً عشان الـ Groupby)
    target_means = y_train.groupby(X_train[col].fillna('UNK')).mean()
    
    # التطبيق على التدريب والاختبار
    X_train[col + '_encoded'] = X_train[col].fillna('UNK').map(target_means).fillna(overall_mean)
    X_test[col + '_encoded'] = X_test[col].fillna('UNK').map(target_means).fillna(overall_mean)
    
    # حذف الأعمدة النصية القديمة
    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)

# 4. One-Hot Encoding للجنس (Sex)
X_train = pd.get_dummies(X_train, columns=['sex'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['sex'], drop_first=False)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 5. تدريب الموديل
print("\n🧠 Training XGBoost Model on 1M+ records (This might take a minute)...")
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=1.40,      # الرقم السحري لضبط التوازن
    max_depth=9,
    learning_rate=0.1,
    n_estimators=450,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=42,
    tree_method='hist',         # بيسرع التدريب جداً مع الداتا الكبيرة
    enable_categorical=False,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# 6. التقييم الصارم على داتا المستقبل (Test Set - Q4)
print("\n🎯 Evaluating on Future Data (Q4 Test Set)...")
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"🌟 ULTIMATE ROC-AUC Score: {auc_score:.4f}\n")

# 7. رسم الـ Confusion Matrix
plt.figure(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Severe (0)', 'Severe (1)'], 
            yticklabels=['Non-Severe (0)', 'Severe (1)'])
plt.title('Confusion Matrix (Out-of-Time Validation)', weight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# 8. Feature Importance (عشان نشوف مين أكتر Feature أثرت في الموديل)
xgb.plot_importance(xgb_model, max_num_features=10, height=0.5, importance_type='weight', title='Top 10 Important Features')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import sqlite3

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

print("🚀 Building MATRIX A (v6) with Patient Medical History (Indications)...")

query_matrix_a_v6 = """
WITH 
-- 1. Target (Severity)
PatientOutcome AS (
    SELECT primaryid, MAX(CASE WHEN outc_cod IN ('DE', 'LT', 'HO', 'DS', 'CA', 'RI') THEN 1 ELSE 0 END) as is_severe
    FROM outc_clean GROUP BY primaryid
),
-- 2. Primary Suspect Drug
PrimaryDrug AS (
    SELECT primaryid, MAX(final_drug_name) as primary_suspect_drug, MAX(route) as ps_route
    FROM drug_clean WHERE role_cod = 'PS' AND final_drug_name != 'none' GROUP BY primaryid
),
-- 3. Polypharmacy
Polypharmacy AS (
    SELECT primaryid, COUNT(final_drug_name) as num_drugs
    FROM drug_clean GROUP BY primaryid
),
-- 4. Therapy Duration
TherapyDuration AS (
    SELECT primaryid, MAX(dur) as therapy_duration
    FROM ther_clean WHERE dur IS NOT NULL GROUP BY primaryid
),
-- 5. Report Source
ReportSource AS (
    SELECT primaryid, MAX(rpsr_cod) as rpsr_cod
    FROM rpsr_clean GROUP BY primaryid
),
-- 6. NEW: Patient Indications (Medical History)
PatientIndications AS (
    SELECT 
        primaryid, 
        COUNT(indi_pt) as num_indications,
        MAX(indi_pt) as primary_indication
    FROM indi_clean 
    WHERE indi_pt IS NOT NULL
    GROUP BY primaryid
)

-- Master Join
SELECT 
    d.primaryid, d.age, d.wt, d.sex, d.occp_cod, d.rept_cod, d.is_test_set,
    p.primary_suspect_drug, p.ps_route,
    COALESCE(poly.num_drugs, 1) as num_drugs,
    t.therapy_duration,
    rs.rpsr_cod,
    COALESCE(ind.num_indications, 0) as num_indications,
    ind.primary_indication,
    COALESCE(o.is_severe, 0) as is_severe
FROM demo_clean d
INNER JOIN PrimaryDrug p ON d.primaryid = p.primaryid
LEFT JOIN Polypharmacy poly ON d.primaryid = poly.primaryid
LEFT JOIN TherapyDuration t ON d.primaryid = t.primaryid
LEFT JOIN ReportSource rs ON d.primaryid = rs.primaryid
LEFT JOIN PatientIndications ind ON d.primaryid = ind.primaryid
LEFT JOIN PatientOutcome o ON d.primaryid = o.primaryid
WHERE d.age IS NOT NULL AND p.primary_suspect_drug IS NOT NULL
"""

df_matrix_a_v6 = pd.read_sql_query(query_matrix_a_v6, conn)
df_matrix_a_v6.to_sql('matrix_a_outcome_v6', conn, if_exists='replace', index=False)

print(f"✅ Matrix A_v6 built successfully! Final Shape: {df_matrix_a_v6.shape}")

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("🚀 Loading Matrix A (v6) and starting ML Pipeline...")
df_ml = pd.read_sql_query("SELECT * FROM matrix_a_outcome_v6", conn)

train_mask = df_ml['is_test_set'] == 0
test_mask = df_ml['is_test_set'] == 1

X_train = df_ml[train_mask].drop(columns=['is_severe', 'is_test_set', 'primaryid'])
y_train = df_ml[train_mask]['is_severe']
X_test = df_ml[test_mask].drop(columns=['is_severe', 'is_test_set', 'primaryid'])
y_test = df_ml[test_mask]['is_severe']

# 💡 تم إضافة primary_indication للـ Encoding
target_cols = ['primary_suspect_drug', 'ps_route', 'rept_cod', 'occp_cod', 'rpsr_cod', 'primary_indication']
overall_mean = y_train.mean()

print("⚙️ Applying Target Encoding...")
for col in target_cols:
    target_means = y_train.groupby(X_train[col].fillna('UNK')).mean()
    X_train[col + '_encoded'] = X_train[col].fillna('UNK').map(target_means).fillna(overall_mean)
    X_test[col + '_encoded'] = X_test[col].fillna('UNK').map(target_means).fillna(overall_mean)
    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)

X_train = pd.get_dummies(X_train, columns=['sex'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['sex'], drop_first=False)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("\n🧠 Training Ultimate XGBoost Model (v6)...")
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=1.40,      
    max_depth=9,
    learning_rate=0.1,
    n_estimators=450,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=42,
    tree_method='hist',         
    enable_categorical=False,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("\n🎯 Evaluating on Future Data (Q4)...")
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"🌟 NEW ULTIMATE ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

xgb.plot_importance(xgb_model, max_num_features=10, height=0.5, importance_type='weight')
plt.title('Top 10 Important Features (Including Indications)')
plt.tight_layout()
plt.show()